In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU available")

CUDA available: True
GPU: Tesla T4


In [6]:
import os

REPO_URL = "https://github.com/realkhaalid/Honours-Year-Project.git"
REPO_NAME = "Honours-Year-Project"
BRANCH = "version_beta"

%cd /content

!rm -rf /content/{REPO_NAME}

!git clone -b {BRANCH} {REPO_URL}

%cd /content/{REPO_NAME}

# Confirm the active branch
!git branch --show-current

# Check files
print("Current directory:", os.getcwd())
print("Files:", os.listdir())

print(
    "data_processing_pipeline.py exists:",
    os.path.exists("data_processing_pipeline.py")
)

/content
Cloning into 'Honours-Year-Project'...
remote: Enumerating objects: 305, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 305 (delta 59), reused 105 (delta 30), pack-reused 171 (from 2)
Receiving objects: 100% (305/305), 76.32 MiB | 23.17 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/Honours-Year-Project
version_beta
Current directory: /content/Honours-Year-Project
Files: ['saved_models', 'README.md', 'archive', 'supervised_fine_tuning_functions.py', '__pycache__', 'unsupervised_reconstruction_functions.py', 'data_processing_pipeline.py', 'train_model_pipeline.py', 'construct_dataset_class.py', 'encoder_functions.py', 'transformer_model_class.py', 'embeddings_functions.py', '.git', 'test_results', 'unsupervised_dataset_class.py']
data_processing_pipeline.py exists: True


In [7]:
import construct_dataset_class

print("Loaded from:")
print(construct_dataset_class.__file__)

print("\nFunctions containing 'dataset':")
for name in dir(construct_dataset_class):
    if "dataset" in name.lower():
        print(name)

print("\nDefined functions in file:")
!grep "^def " /content/Honours-Year-Project/construct_dataset_class.py

Loaded from:
/content/Honours-Year-Project/construct_dataset_class.py

Functions containing 'dataset':
Dataset
SupervisedAudioDataset
check_supervised_dataset
create_datasets

Defined functions in file:
def collect_supervised_audio_files(
def create_label_mapping(
def split_training_files(
def create_datasets(
def check_supervised_dataset(
def check_data_loader(


In [4]:
from torch.utils.data import DataLoader
from data_processing_pipeline import (
    convert_to_mel_spectrogram,
    convert_to_stft_spectrogram,
    convert_to_cqt_spectrogram
)
from construct_dataset_class import (
    create_train_val_datasets,
    SLAKH2100_REDUX_16K_TRAIN,
    SLAKH2100_REDUX_16K_VALIDATION
)
from train_model_pipeline import (
    TrainModelPipeline,
    EMBEDDING_DIM,
    NUM_HEADS,
    HIDDEN_DIMS,
    NUM_ENCODER_LAYERS,
    MASK_RATIO,
    ACTIVATION,
    CHECKPOINT_INTERVAL,
    CHECKPOINT_DIRECTORY,
    UNSUPERVISED_EPOCHS,
    SUPERVISED_EPOCHS,
    UNSUPERVISED_LEARNING_RATE,
    SUPERVISED_LEARNING_RATE,
    PRETRAINED_MODEL_NAME,
    SUPERVISED_ONLY_MODEL_NAME
)

DATA_REPRESENTATION = convert_to_mel_spectrogram
BATCH_SIZE = 32

# Create Datasets
print("Creating datasets...")
(
    supervised_training_dataset,
    supervised_validation_dataset,
    unsupervised_training_dataset,
    unsupervised_validation_dataset,
    label_to_index_val
) = create_train_val_datasets(
    training_path=SLAKH2100_REDUX_16K_TRAIN,
    validation_path=SLAKH2100_REDUX_16K_VALIDATION,
    data_representation=DATA_REPRESENTATION,
    set_limit=False
)

# Retrieve model label mappings
(
    label_to_index,
    index_to_label
) = supervised_training_dataset.return_label_mappings()

num_classes = len(label_to_index)

# Check datasets
print("\nDataset Information")
print("=" * 70)

print(
    "Unsupervised training dataset size:",
    len(unsupervised_training_dataset)
)

print(
    "Unsupervised validation dataset size:",
    len(unsupervised_validation_dataset)
)

print(
    "Supervised training dataset size:",
    len(supervised_training_dataset)
)

print(
    "Supervised validation dataset size:",
    len(supervised_validation_dataset)
)

print(
    "Label to index mapping:",
    label_to_index
)

print(
    "Index to label mapping:",
    index_to_label
)

print(
    "Number of classes:",
    num_classes
)

print(
    "Training mapping matches validation mapping:",
    label_to_index == label_to_index_val
)

# DataLoaders
unsupervised_training_loader = DataLoader(
    unsupervised_training_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

unsupervised_validation_loader = DataLoader(
    unsupervised_validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

supervised_training_loader = DataLoader(
    supervised_training_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

supervised_validation_loader = DataLoader(
    supervised_validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Get sample batch for Transformer initialization
(
    sample_batch,
    _
) = next(
    iter(supervised_training_loader)
)

# Train and evaluate the model
training_pipeline = TrainModelPipeline(
    embedding_dim=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    hidden_dims=HIDDEN_DIMS,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_classes=num_classes,
    mask_ratio=MASK_RATIO,
    activation=ACTIVATION,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    checkpoint_directory=CHECKPOINT_DIRECTORY
)

print(
    "\nTraining device:",
    training_pipeline.device
)

print(
    "CUDA available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

# Unsupervised pretraining + supervised fine-tuning
# print("\n")
# print("#" * 70)

# print(
#     "MODEL 1: UNSUPERVISED PRETRAINING "
#     "FOLLOWED BY SUPERVISED FINE-TUNING"
# )

# print("#" * 70)

# pretrained_results = (
#     training_pipeline
#     .train_pretrained_and_fine_tuned_model(
#         sample_batch=sample_batch,
#         unsupervised_training_loader=unsupervised_training_loader,
#         unsupervised_validation_loader=unsupervised_validation_loader,
#         supervised_training_loader=supervised_training_loader,
#         supervised_validation_loader=supervised_validation_loader,
#         unsupervised_epochs=UNSUPERVISED_EPOCHS,
#         supervised_epochs=SUPERVISED_EPOCHS,
#         unsupervised_learning_rate=UNSUPERVISED_LEARNING_RATE,
#         supervised_learning_rate=SUPERVISED_LEARNING_RATE,
#         model_name=PRETRAINED_MODEL_NAME,
#         label_to_index=label_to_index,
#         index_to_label=index_to_label
#     )
# )

# # Retrieve trained model
# pretrained_fine_tuned_model = pretrained_results["model"]

# # Print unsupervised training history
# print("\nUnsupervised Pretraining")
# print("=" * 70)

# for epoch_results in pretrained_results["unsupervised_results"]["history"]:
#     print(
#         f"Epoch {epoch_results['epoch']} | "
#         f"Training loss: {epoch_results['training_loss']:.6f} | "
#         f"Validation loss: {epoch_results['validation_loss']:.6f}"
#     )

# print(
#     "\nBest unsupervised validation loss:",
#     pretrained_results["unsupervised_results"]["best_validation_loss"]
# )

# print(
#     "Best unsupervised epoch:",
#     pretrained_results["unsupervised_results"]["best_epoch"]
# )

# # Print supervised fine-tuning history
# print("\nSupervised Fine-Tuning")
# print("=" * 70)

# for epoch_results in pretrained_results["supervised_results"]["history"]:
#     print(
#         f"Epoch {epoch_results['epoch']} | "
#         f"Training loss: {epoch_results['training_loss']:.6f} | "
#         f"Training accuracy: "
#         f"{epoch_results['training_accuracy'] * 100:.2f}% | "
#         f"Validation loss: {epoch_results['validation_loss']:.6f} | "
#         f"Validation accuracy: "
#         f"{epoch_results['validation_accuracy'] * 100:.2f}%"
#     )

# print(
#     "\nBest supervised validation loss:",
#     pretrained_results["supervised_results"]["best_validation_loss"]
# )

# print(
#     "Best supervised validation accuracy:",
#     pretrained_results["supervised_results"]["best_validation_accuracy"] * 100
# )

# print(
#     "Best supervised epoch:",
#     pretrained_results["supervised_results"]["best_epoch"]
# )

# Supervised-only training
print("\n")
print("#" * 70)

print(
    "MODEL 2: SUPERVISED-ONLY TRAINING"
)

print("#" * 70)

supervised_only_results = (
    training_pipeline
    .train_supervised_only_model(
        sample_batch=sample_batch,
        supervised_training_loader=supervised_training_loader,
        supervised_validation_loader=supervised_validation_loader,
        epochs=SUPERVISED_EPOCHS,
        learning_rate=SUPERVISED_LEARNING_RATE,
        model_name=SUPERVISED_ONLY_MODEL_NAME,
        label_to_index=label_to_index,
        index_to_label=index_to_label
    )
)

# Retrieve supervised-only model
supervised_only_model = supervised_only_results["model"]

# Print supervised-only training history
print("\nSupervised-Only Training")
print("=" * 70)

for epoch_results in supervised_only_results["history"]:
    print(
        f"Epoch {epoch_results['epoch']} | "
        f"Training loss: {epoch_results['training_loss']:.6f} | "
        f"Training accuracy: "
        f"{epoch_results['training_accuracy'] * 100:.2f}% | "
        f"Validation loss: {epoch_results['validation_loss']:.6f} | "
        f"Validation accuracy: "
        f"{epoch_results['validation_accuracy'] * 100:.2f}%"
    )

print(
    "\nBest supervised-only validation loss:",
    supervised_only_results["best_validation_loss"]
)

print(
    "Best supervised-only validation accuracy:",
    supervised_only_results["best_validation_accuracy"] * 100
)

print(
    "Best supervised-only epoch:",
    supervised_only_results["best_epoch"]
)

ImportError: cannot import name 'create_train_val_datasets' from 'construct_dataset_class' (/content/Honours-Year-Project/construct_dataset_class.py)